In [1]:
import pandas as pd

df = pd.read_csv("..\\data\\raw\\dataset_csv.csv", sep=r"\s+", engine="python")

df.head()

,obs_id,class,default,x1,x2,x3,x4,x5,x6,x7,...,x21,x22,x23,x24,x25,x26,time,year,testing_set,training_set
0,0,1406,0,0.548690,0.378213,0.516406,1.608842,0.602419,0.065173,0.558509,...,0.537593,0.001444,0.382384,0.033177,1.000000,0,1,2007,1,0
1,1,1406,0,0.547813,0.378383,0.516522,1.699176,0.601911,0.062662,0.557950,...,0.572315,0.001311,0.380498,0.028798,1.000000,0,2,2008,1,0
2,2,1406,0,0.549332,0.378818,0.514555,1.734131,0.603207,0.063617,0.559306,...,0.583454,0.001697,0.381134,0.030690,1.000000,0,3,2009,1,0
3,3,1406,0,0.552237,0.379319,0.512808,1.852136,0.605443,0.078868,0.565012,...,0.583256,0.001229,0.368001,0.062207,0.903143,0,4,2010,1,0
4,4,1406,0,0.550137,0.379200,0.515377,1.750650,0.603519,0.084289,0.560092,...,0.569079,0.001034,0.368143,0.073117,0.922740,0,5,2011,1,0


In [2]:
train_df = df[df["training_set"] == 1]

test_df = df[df["testing_set"] == 1]

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (2961, 33)
Test shape: (1250, 33)


In [3]:

train_df = train_df.drop(columns=["training_set", "testing_set"])
test_df = test_df.drop(columns=["training_set", "testing_set"])

train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

train_df.head()

,obs_id,class,default,x1,x2,x3,x4,x5,x6,x7,...,x19,x20,x21,x22,x23,x24,x25,x26,time,year
0,11,1547,0,0.544009,0.380165,0.525489,0.692744,0.599246,0.099967,0.555989,...,0.019701,0.401271,0.595331,0.000091,0.371238,0.020211,1.000000,0,1,2007
1,12,1547,0,0.544032,0.380000,0.528304,0.668594,0.599254,0.096937,0.555563,...,0.019701,0.404526,0.591292,0.000056,0.373153,0.018726,1.000000,0,2,2008
2,13,1547,0,0.545822,0.380194,0.523855,0.716922,0.600820,0.097128,0.561278,...,0.019701,0.399808,0.591566,0.000168,0.372171,0.018761,1.000000,0,3,2009
3,14,1547,0,0.543440,0.380163,0.525681,0.656266,0.598775,0.084725,0.554249,...,0.115096,0.395659,0.586824,0.000196,0.372179,0.079319,1.000000,0,4,2010
4,15,1547,0,0.543460,0.380257,0.523464,0.655786,0.598800,0.081741,0.554244,...,0.101897,0.392870,0.590423,0.000162,0.367481,0.072141,0.921513,0,5,2011


In [4]:
# Cell 4 — Explore time / year mapping
print("Max time value:", df["time"].max())
print()
print("Unique (time, year) pairs:")
df[["time", "year"]].drop_duplicates().sort_values("time")

Max time value: 11

Unique (time, year) pairs:


,time,year
0,1,2007
22,1,2009
41,1,2010
37,1,2012
47,1,2008
...,...,...
55,9,2016
8,9,2015
56,10,2017
9,10,2016


In [5]:
# Cell 5 — Verify time spans per company
df.groupby("time")["year"].unique()

time
1     [2007, 2009, 2012, 2010, 2008, 2011, 2013, 201...
2     [2008, 2010, 2013, 2011, 2009, 2012, 2014, 201...
3     [2009, 2011, 2014, 2012, 2010, 2013, 2015, 201...
4      [2010, 2012, 2015, 2013, 2011, 2014, 2016, 2017]
5            [2011, 2013, 2014, 2012, 2016, 2015, 2017]
6                  [2012, 2014, 2015, 2013, 2017, 2016]
7                        [2013, 2015, 2014, 2016, 2017]
8                              [2014, 2016, 2015, 2017]
9                                    [2015, 2017, 2016]
10                                         [2016, 2017]
11                                               [2017]
Name: year, dtype: object

In [6]:
# Cell 6 — Inspect a subset of the panel structure
df[["class", "default", "time", "year", "training_set", "testing_set"]].iloc[:100]

,class,default,time,year,training_set,testing_set
0,1406,0,1,2007,0,1
1,1406,0,2,2008,0,1
2,1406,0,3,2009,0,1
3,1406,0,4,2010,0,1
4,1406,0,5,2011,0,1
...,...,...,...,...,...,...
95,10863,0,5,2012,0,1
96,10863,0,6,2013,0,1
97,10863,0,7,2014,0,1
98,10863,0,8,2015,0,1


In [7]:
# Cell 7 — Drop correlated features (identified during EDA)
# x7, x19, x25 were dropped due to high correlation with other features.
# x6, x24 were dropped in the original analysis as well.
# class is a non-informative label column.
# obs_id is KEPT — needed by the LSTM for per-company sequencing.

train_df = train_df.rename(columns={"class": "company_id"})
test_df  = test_df.rename(columns={"class": "company_id"})

COLS_TO_DROP_MODEL = ["x6", "x7", "x19", "x24", "x25"]

train_df = train_df.drop(columns=COLS_TO_DROP_MODEL, errors="ignore")
test_df  = test_df.drop(columns=COLS_TO_DROP_MODEL, errors="ignore")

print("Train columns:", list(train_df.columns))
print("\nTest columns: ", list(test_df.columns))

Train columns: ['obs_id', 'company_id', 'default', 'x1', 'x2', 'x3', 'x4', 'x5', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15', 'x16', 'x17', 'x18', 'x20', 'x21', 'x22', 'x23', 'x26', 'time', 'year']

Test columns:  ['obs_id', 'company_id', 'default', 'x1', 'x2', 'x3', 'x4', 'x5', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15', 'x16', 'x17', 'x18', 'x20', 'x21', 'x22', 'x23', 'x26', 'time', 'year']


In [8]:
# Cell 8 — Verify shapes and column list
print(f"Train shape : {train_df.shape}")
print(f"Test shape  : {test_df.shape}")
print()
print("Columns in saved files:")
for col in train_df.columns:
    role = "(target)" if col == "default" else \
           "(metadata — excluded from model features)" if col in ["obs_id", "time", "year"] else \
           "(feature)"
    print(f"  {col:10s}  {role}")

Train shape : (2961, 26)
Test shape  : (1250, 26)

Columns in saved files:
  obs_id      (metadata — excluded from model features)
  company_id  (feature)
  default     (target)
  x1          (feature)
  x2          (feature)
  x3          (feature)
  x4          (feature)
  x5          (feature)
  x8          (feature)
  x9          (feature)
  x10         (feature)
  x11         (feature)
  x12         (feature)
  x13         (feature)
  x14         (feature)
  x15         (feature)
  x16         (feature)
  x17         (feature)
  x18         (feature)
  x20         (feature)
  x21         (feature)
  x22         (feature)
  x23         (feature)
  x26         (feature)
  time        (metadata — excluded from model features)
  year        (metadata — excluded from model features)


In [9]:
# Cell 9 — Save processed data
# obs_id, time, and year are kept as metadata columns.
# CatBoost notebook drops them when building X_train / X_test.
# LSTM notebook uses obs_id to group sequences per company.

train_df.to_csv('../data/processed/features_train.csv', index=False)
test_df.to_csv('../data/processed/features_test.csv', index=False)

print(f"Saved features_train.csv  — shape: {train_df.shape}")
print(f"Saved features_test.csv   — shape: {test_df.shape}")
print("\nNote: obs_id / time / year are metadata, not model inputs.")
print("They will be excluded from X when building feature matrices.")

Saved features_train.csv  — shape: (2961, 26)
Saved features_test.csv   — shape: (1250, 26)

Note: obs_id / time / year are metadata, not model inputs.
They will be excluded from X when building feature matrices.
